# Osonye Onyemazuwa — Week 2: Database Setup
### Day 9: Create the database (PostgreSQL); load cleaned Ad Spend data

**Issue #32:** Create the database (PostgreSQL/SQLite); load cleaned Ad Spend data.

**Note:** this notebook was written and schema-checked against the Day 8
design, but not executed against a live PostgreSQL server in this
environment (no server/network access here). Run this locally against
your own Postgres instance and flag anything that errors.

### Requirements
```
pip install psycopg2-binary sqlalchemy python-dotenv
```
You will need a running PostgreSQL server and a database created first, e.g.:
```
create db attribution
```

### Credentials setup (do this once, before running the cells below)
Create a `.env` file in this same folder (never commit this file) with:
```
DB_USER=postgres
DB_PASSWORD=your_actual_password
DB_HOST=localhost
DB_PORT=5432
DB_NAME=attribution
```
Then confirm `.env` is listed in `.gitignore` before running anything below.


In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install psycopg2-binary

In [ ]:
import os
import urllib.parse
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Credentials loaded from .env (never hardcoded, never committed -- see .gitignore)
load_dotenv()

DB_USER = os.environ.get("DB_USER")
DB_PASSWORD = urllib.parse.quote_plus(os.environ.get("DB_PASSWORD")) # encloses special characters like @ 
DB_HOST = os.environ.get("DB_HOST")
DB_PORT = os.environ.get("DB_PORT")
DB_NAME = os.environ.get("DB_NAME")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

print("Connected as:", DB_USER)  # safe to print -- never print DB_PASSWORD


## Create tables (schema from Day 8)

In [ ]:
create_campaigns = text("""
CREATE TABLE IF NOT EXISTS campaigns (
    campaign_id      INTEGER PRIMARY KEY,
    channel          TEXT NOT NULL,
    objective        TEXT,
    start_date       DATE,
    end_date         DATE,
    target_segment   TEXT,
    expected_uplift  NUMERIC
);
""")

create_ad_spend = text("""
CREATE TABLE IF NOT EXISTS ad_spend (
    spend_id       TEXT PRIMARY KEY,
    date           DATE NOT NULL,
    campaign_id    INTEGER NOT NULL REFERENCES campaigns(campaign_id),
    channel        TEXT NOT NULL,
    utm_source     TEXT,
    utm_medium     TEXT,
    utm_campaign   TEXT,
    impressions    INTEGER,
    clicks         INTEGER,
    ad_spend       NUMERIC NOT NULL,
    currency       TEXT
);
""")

with engine.begin() as conn:
    conn.execute(create_campaigns)
    conn.execute(create_ad_spend)

print("Tables created (or already existed).")


**Note on table order:** `ad_spend.campaign_id` has a `REFERENCES
campaigns(campaign_id)` foreign key, so `campaigns` must be created (and
loaded with data) before any rows can be inserted into `ad_spend`,
otherwise Postgres will reject the insert with a foreign key violation.


## Load cleaned data

## Clear tables before loading (safe to re-run)

Running this notebook more than once will otherwise try to insert rows
that already exist (since we use `append`, not `replace`, to protect the
foreign key constraints -- see note above). This cell empties both
tables first so every re-run starts clean.


In [ ]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE ad_spend, campaigns RESTART IDENTITY CASCADE;"))

print("Tables cleared for fresh load.")


In [ ]:
# Campaigns: load and standardize channel (loaded first -- ad_spend references it)
campaigns = pd.read_csv("campaigns.csv")
campaigns['channel'] = campaigns['channel'].str.strip().str.lower().str.replace(' ', '_', regex=False)

campaigns.to_sql('campaigns', engine, if_exists='append', index=False)


In [ ]:
# Ad spend: load and standardize channel/utm_source
ad_spend = pd.read_csv("ad_spend.csv")
ad_spend['channel'] = ad_spend['channel'].str.strip().str.lower().str.replace(' ', '_', regex=False)
ad_spend['utm_source'] = ad_spend['utm_source'].str.strip().str.lower()

ad_spend.to_sql('ad_spend', engine, if_exists='append', index=False)

print("Loaded.")


**Resolved:** switched to `if_exists='append'` since the tables already exist with the correct schema/constraints from the CREATE TABLE step above. Using `replace` here would try to DROP the `campaigns` table, which fails once `ad_spend` has a foreign key pointing to it. The TRUNCATE cell above handles safe re-runs instead.


## Verify: row counts match source files (no data lost on load)

In [ ]:
with engine.connect() as conn:
    campaigns_count = conn.execute(text("SELECT COUNT(*) FROM campaigns")).scalar()
    ad_spend_count = conn.execute(text("SELECT COUNT(*) FROM ad_spend")).scalar()

print("campaigns rows in DB:", campaigns_count, "  | source CSV:", len(campaigns))
print("ad_spend rows in DB:", ad_spend_count, "  | source CSV:", len(ad_spend))


## Verify: spend-by-channel matches Week 1's pandas result

In [ ]:
query = text("""
SELECT channel, ROUND(SUM(ad_spend), 2) as total_spend, COUNT(*) as rows
FROM ad_spend
GROUP BY channel
ORDER BY total_spend DESC
""")

with engine.connect() as conn:
    result = conn.execute(query)
    for row in result:
        print(row)


Expected to match Week 1's pandas output exactly (Affiliate **\$6,454.96** →
Social **\$4,176.33**) — run this locally and confirm it matches before moving on.


## Verify: join between ad_spend and campaigns works

In [ ]:
query = text("""
SELECT COUNT(*) FROM ad_spend a
JOIN campaigns c ON a.campaign_id = c.campaign_id
""")

with engine.connect() as conn:
    joined_count = conn.execute(query).scalar()

print("Joined rows:", joined_count, "  | expected (all rows):", len(ad_spend))


Expected: all 2,599 rows join successfully (matches Day 3's finding that
campaign_id integrity is clean, no orphans). Confirm locally.

## Day 9 notes

- Using PostgreSQL (not SQLite) per team decision
- Successfully run and verified locally against a real Postgres instance
- Password contains a special character (@) -- had to URL-encode it with
  urllib.parse.quote_plus() or SQLAlchemy misreads it as part of the host
- Had to manually create the attribution database first (CREATE DATABASE
  attribution;) -- create_engine() doesn't create it automatically
- Hit a foreign key conflict using to_sql(if_exists='replace') -- Postgres
  refuses to drop campaigns while ad_spend references it. Fixed by
  switching to if_exists='append' and adding a TRUNCATE cell to make
  re-runs safe
- Credentials loaded from .env (never hardcoded/committed) -- see
  .gitignore
- Loaded campaigns before ad_spend since ad_spend has a FK reference
- Same channel/utm_source standardization applied as Week 1
